# PCMCI + FAISS CMI — All Crops, Monthly

Runs the full preprocessing pipeline and PCMCI + FAISS-backed CMI on **corn, soybean, and wheat** at monthly frequency.

Three feature sets per crop:
- `climate` — 10 climate variables only
- `climate_rv` — climate + realized volatility
- `macro_news_climate_rv` — full model (all 18 features)

**Why CMI / FAISSCMI:**
- CMI = I(X;Y|Z) is the most general nonparametric CI test — detects any statistical dependency (linear, nonlinear, multiplicative, non-additive) without distributional assumptions.
- FAISS-backed k-NN replaces brute-force neighbor search with an optimized L2 index, giving 3–10× faster joint-space searches compared to CMIknn's sklearn backend.
- Unlike GPDC (additive noise assumption) and ParCorr (linear + Gaussian), CMI detects multiplicative / threshold-type dependencies that both miss.

**Tradeoff:** CMI requires a shuffle test (no analytic null), so each CI test runs `sig_samples` additional CMI computations. Full PCMCI runs are substantially slower than ParCorr or GPDC.

Preprocessing identical to `eda_preprocessing.ipynb`: interpolate → ffill → Gaussian detrend (15-yr) → anomalize → `pp.trafo2normal`.

## 0. Imports & Configuration

In [1]:
import os
# Must be set before importing faiss or MKL-linked numpy
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')   # macOS: prevents duplicate libiomp5 crash
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('VECLIB_MAXIMUM_THREADS', '1')

import sys
from pathlib import Path
from typing import Any

ROOT = Path('.').resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy.special import digamma

import faiss
from tigramite import data_processing as pp
from tigramite import plotting as tp
from tigramite.pcmci import PCMCI
from tigramite.independence_tests.independence_tests_base import CondIndTest

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'legend.fontsize': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# ── Configuration ─────────────────────────────────────────────────────────────
CROPS     = ['corn', 'soybean', 'wheat']
FREQUENCY = 'monthly'

CYCLE_LEN    = 12
SMOOTH_WIDTH = 15 * CYCLE_LEN   # 15-year Gaussian detrend window

TAU_MAX          = 3     # shorter than ParCorr: every MCI test runs sig_samples shuffles
PC_ALPHA         = 0.05  # tighter skeleton → fewer expensive shuffle-test MCI evaluations
ALPHA_LEVEL      = 0.05
FDR_METHOD       = 'fdr_bh'

K_NEIGHBORS      = 5     # Kraskov et al. recommend k in [3, 10]; k=5 is robust for T≈192
SIG_SAMPLES      = 200   # shuffle samples per link test; matches GPDC project standard
MAX_COMBINATIONS = 1     # test only best conditioning set per MCI link (runtime control)
RANDOM_SEED      = 42

print(f'Crops            : {CROPS}')
print(f'Frequency        : {FREQUENCY}')
print(f'tau_max          : {TAU_MAX}')
print(f'pc_alpha         : {PC_ALPHA}')
print(f'k_neighbors      : {K_NEIGHBORS}')
print(f'sig_samples      : {SIG_SAMPLES}')
print(f'max_combinations : {MAX_COMBINATIONS}')

Crops            : ['corn', 'soybean', 'wheat']
Frequency        : monthly
tau_max          : 3
pc_alpha         : 0.05
k_neighbors      : 5
sig_samples      : 200
max_combinations : 1


## 1. Backend Detection

FAISS supports **CUDA GPU** and **CPU** only — it does **not** use MPS (Apple Silicon).
On macOS M-series, FAISSCMI falls back to CPU automatically.

In [2]:
_faiss_gpu_count = 0
FAISS_USE_GPU    = False
try:
    _get_num = getattr(faiss, 'get_num_gpus', None)
    if _get_num is not None:
        _faiss_gpu_count = int(_get_num())
        FAISS_USE_GPU = _faiss_gpu_count > 0
except Exception:
    pass

_cuda_available = False
_mps_available  = False
try:
    import torch
    _cuda_available = torch.cuda.is_available()
    _mps_back = getattr(getattr(torch, 'backends', None), 'mps', None)
    _mps_available = _mps_back is not None and _mps_back.is_available()
except ImportError:
    pass

print(f'faiss version    : {faiss.__version__}')
print(f'FAISS GPU count  : {_faiss_gpu_count}')
print(f'FAISS GPU active : {FAISS_USE_GPU}')
print(f'CUDA available   : {_cuda_available}')
print(f'MPS available    : {_mps_available}  (FAISS does not use MPS)')
print()
if FAISS_USE_GPU:
    print('Backend: CUDA GPU — k-NN search will use GPU device 0.')
elif _mps_available:
    print('Backend: CPU — MPS detected but FAISS has no MPS backend.')
    print('Runtime estimate: 2–8 min per (crop x feature-set) at tau_max=3, sig_samples=200.')
else:
    print('Backend: CPU')
    print('Runtime estimate: 2–8 min per (crop x feature-set) at tau_max=3, sig_samples=200.')

faiss version    : 1.14.1
FAISS GPU count  : 0
FAISS GPU active : False
CUDA available   : False
MPS available    : True  (FAISS does not use MPS)

Backend: CPU — MPS detected but FAISS has no MPS backend.
Runtime estimate: 2–8 min per (crop x feature-set) at tau_max=3, sig_samples=200.


## 2. FAISSCMI — FAISS-backed CMI Estimator

**Kraskov et al. 2004 estimator (variant 1):**

$$I(X;Y|Z) = \psi(k) + \langle\psi(n_z + 1)\rangle - \langle\psi(n_{xz} + 1) + \psi(n_{yz} + 1)\rangle$$

where $\psi$ is the digamma function, $k$ is the number of nearest neighbors, and $n_{xz}$, $n_{yz}$, $n_z$ count neighbors in the marginal subspaces within the $k$-th-neighbor radius $\varepsilon_i$ found in the full joint $(x, y, z)$ space.

**FAISS role:** `IndexFlatL2` handles the joint-space k-NN search (`_knn_distances`). Marginal counting (`_count_neighbors`) uses NumPy — O(T²) per call, acceptable for T ≤ 300 monthly observations.

**Null distribution:** Shuffle test — X is permuted block-wise `sig_samples` times and CMI is recomputed each time. The block structure preserves short-range autocorrelation in the null. No analytic null exists for CMI.

In [3]:
class FAISSCMI(CondIndTest):
    """
    FAISS-backed Conditional Mutual Information CI test for PCMCI.

    Estimates I(X;Y|Z) using the Kraskov et al. 2004 k-NN estimator with
    FAISS IndexFlatL2 for efficient joint-space neighbor search. Marginal
    neighbor counting uses NumPy (O(T^2) per test — adequate for T <= 300).

    Parameters
    ----------
    k : int
        Number of nearest neighbors. Kraskov et al. recommend 3-10; k=5 is
        robust for T ~ 200. Higher k reduces variance but increases bias.
    use_gpu : bool
        Attempt FAISS GPU index (CUDA only, not MPS). Falls back to CPU.
    gpu_device : int
        CUDA device index (only when use_gpu=True and CUDA is present).
    standardize : bool
        Z-standardize each variable so the k-NN radius is comparably sized
        across variables with different scales.
    jitter : float
        Tiny Gaussian noise added after standardization to break k-NN ties
        that would bias the CMI estimate upward.
    significance : str
        Must be 'shuffle_test'. CMI has no analytic null distribution.
    **kwargs
        Forwarded to CondIndTest (seed, sig_samples, sig_blocklength,
        verbosity, ...).

    Notes
    -----
    FAISS GPU resources (StandardGpuResources) are SWIG-backed and cannot be
    pickled. __getstate__ / __setstate__ drop and rebuild the GPU handle so
    the object survives serialization (e.g. joblib parallelism in PCMCI).
    """

    X_ID = 0
    Y_ID = 1
    Z_ID = 2

    @property
    def measure(self) -> Any:
        return self._measure

    def __init__(
        self,
        k: int = 5,
        *,
        use_gpu: bool = True,
        gpu_device: int = 0,
        standardize: bool = True,
        jitter: float = 1e-6,
        significance: str = "shuffle_test",
        **kwargs: Any,
    ) -> None:
        if k < 1:
            raise ValueError("k must be >= 1.")
        if significance == "analytic":
            raise ValueError(
                "FAISSCMI does not support analytic significance; use 'shuffle_test'."
            )
        self.k                 = int(k)
        self.use_gpu           = use_gpu
        self.gpu_device        = gpu_device
        self.standardize       = standardize
        self.jitter            = jitter
        self._measure          = "faiss_cmi"
        self.two_sided         = False
        self.residual_based    = False
        self.recycle_residuals = False
        self.res = self._build_gpu_resources() if use_gpu else None
        CondIndTest.__init__(self, significance=significance, **kwargs)

    # ── Pickling ──────────────────────────────────────────────────────────────
    def __getstate__(self) -> dict:
        state = self.__dict__.copy()
        state["res"] = None   # FAISS SWIG pointer — not picklable
        return state

    def __setstate__(self, state: dict) -> None:
        self.__dict__.update(state)
        self.res = self._build_gpu_resources() if self.use_gpu else None

    # ── FAISS index management ────────────────────────────────────────────────
    def _build_gpu_resources(self) -> Any | None:
        std_gpu = getattr(faiss, "StandardGpuResources", None)
        if std_gpu is None:
            return None
        try:
            return std_gpu()
        except Exception:
            return None

    def _build_index(self, dim: int) -> Any:
        cpu_index = faiss.IndexFlatL2(dim)
        to_gpu    = getattr(faiss, "index_cpu_to_gpu", None)
        if self.res is None or to_gpu is None:
            return cpu_index
        try:
            return to_gpu(self.res, self.gpu_device, cpu_index)
        except Exception:
            return cpu_index

    # ── Data preparation ──────────────────────────────────────────────────────
    def _prepare_array(self, array: np.ndarray) -> np.ndarray:
        """Z-standardize then add small jitter to prevent k-NN ties."""
        prepared = np.asarray(array, dtype=np.float64).copy()
        if self.standardize:
            prepared -= prepared.mean(axis=1, keepdims=True)
            std = prepared.std(axis=1, keepdims=True)
            std[std == 0.0] = 1.0
            prepared /= std
        if self.jitter > 0.0:
            scale = prepared.std(axis=1, keepdims=True)
            scale[scale == 0.0] = 1.0
            prepared += self.jitter * scale * self.random_state.random(prepared.shape)
        return prepared

    # ── k-NN search (FAISS) ───────────────────────────────────────────────────
    def _knn_distances(self, data: np.ndarray) -> np.ndarray:
        """
        Return the L2 distance to the k-th nearest neighbor for each sample.
        This radius eps_i drives marginal neighbor counting in the Kraskov formula.
        """
        samples = np.ascontiguousarray(data, dtype=np.float32)
        index   = self._build_index(samples.shape[1])
        index.add(samples)
        distances, _ = index.search(samples, self.k + 1)
        # distances[:, 0] = 0 (self-distance); [:, k] = k-th neighbor distance
        return np.sqrt(distances[:, -1].astype(np.float64))

    def _count_neighbors(self, space: np.ndarray, eps: np.ndarray) -> np.ndarray:
        """
        Count samples strictly within radius eps_i in a marginal subspace.
        Returns n_xz, n_yz, or n_z depending on the subspace passed.
        O(T^2) per call — the main runtime bottleneck for large T.
        """
        n = space.shape[0]
        if space.shape[1] == 0:
            # Empty Z (unconditional MI): every other point is a neighbor
            return np.full(n, n - 1, dtype=np.float64)
        counts = np.zeros(n, dtype=np.float64)
        for i in range(n):
            dists     = np.linalg.norm(space - space[i], axis=1)
            counts[i] = np.sum(dists < eps[i]) - 1   # exclude self
        return counts

    # ── CMI estimation ────────────────────────────────────────────────────────
    def get_dependence_measure(
        self,
        array: np.ndarray,
        xyz: np.ndarray,
        data_type: np.ndarray | None = None,
    ) -> float:
        """
        Kraskov 2004 estimator variant 1:

            I(X;Y|Z) = psi(k)
                     + mean( psi(n_z  + 1) )
                     - mean( psi(n_xz + 1) + psi(n_yz + 1) )

        array : shape (D, T) — rows are variables, columns are time points
        xyz   : shape (D,)  — role labels: 0=X, 1=Y, 2=Z
        """
        del data_type
        prepared  = self._prepare_array(array)
        x = prepared[xyz == self.X_ID].T   # (T, dx)
        y = prepared[xyz == self.Y_ID].T   # (T, dy)
        z = prepared[xyz == self.Z_ID].T   # (T, dz)

        xyz_joint = np.column_stack([x, y, z])
        xz_space  = np.column_stack([x, z])
        yz_space  = np.column_stack([y, z])

        eps  = self._knn_distances(xyz_joint)
        n_xz = self._count_neighbors(xz_space, eps)
        n_yz = self._count_neighbors(yz_space, eps)
        n_z  = self._count_neighbors(z, eps)

        return float(
            digamma(self.k)
            + np.mean(digamma(n_z  + 1.0))
            - np.mean(digamma(n_xz + 1.0) + digamma(n_yz + 1.0))
        )

    # ── Significance testing ──────────────────────────────────────────────────
    def get_shuffle_significance(
        self,
        array: np.ndarray,
        xyz: np.ndarray,
        value: float,
        data_type: np.ndarray | None = None,
        return_null_dist: bool = False,
    ) -> Any:
        """
        Block-shuffle X to build a CMI null distribution under H0: X ind Y | Z.

        Each of sig_samples shuffles permutes the X rows block-wise (block
        length = sig_blocklength), preserving short-range autocorrelation in
        the shuffled X. Falls back to block=1 if the block length is too large.

        p-value = (# null >= observed + 1) / (sig_samples + 1)
        The +1 is a conservative correction for finite shuffle samples.
        """
        del data_type
        try:
            null_dist = self._get_shuffle_dist(
                array, xyz, self.get_dependence_measure,
                sig_samples=self.sig_samples,
                sig_blocklength=self.sig_blocklength,
                verbosity=self.verbosity,
            )
        except (TypeError, ValueError):
            null_dist = self._get_shuffle_dist(
                array, xyz, self.get_dependence_measure,
                sig_samples=self.sig_samples,
                sig_blocklength=1,
                verbosity=self.verbosity,
            )
        pval = float(np.sum(null_dist >= value) + 1) / (self.sig_samples + 1)
        if return_null_dist:
            return pval, null_dist
        return pval


print("FAISSCMI class defined.")
print("  measure     : faiss_cmi")
print("  two_sided   : False  (CMI >= 0; one-sided test)")
print("  significance: shuffle_test  (only valid option)")

FAISSCMI class defined.
  measure     : faiss_cmi
  two_sided   : False  (CMI >= 0; one-sided test)
  significance: shuffle_test  (only valid option)


## 3. Feature Group Definitions

| Group | Variables | N |
|-------|-----------|---|
| `rv` | `weekly_rv` | 1 |
| `climate` | `prcp`, `awnd`, `tmin`, `tmax`, `co2`, `pdsi`, `EVAP`, `soil_moisture_mean_m3m3`, `cloud_cover_mean_pct`, `humidity_mean_pct` | 10 |
| `macro` | `DJIA_Index`, `WTI_Index`, `Broad_Dollar_index`, `Stock_Uncertainty` | 4 |
| `news` | `frbsf_sentiment`, `epu_index`, `Text_Climate_Anomaly` | 3 |

**Feature sets run per crop:**

| Feature Set | Groups | N |
|-------------|--------|---|
| `climate` | climate | 10 |
| `climate_rv` | climate + rv | 11 |
| `macro_news_climate_rv` | macro + news + climate + rv | 18 |

In [4]:
FEATURE_GROUPS = {
    'rv'     : ['weekly_rv'],
    'climate': ['prcp', 'awnd', 'tmin', 'tmax', 'co2', 'pdsi',
                'EVAP', 'soil_moisture_mean_m3m3', 'cloud_cover_mean_pct', 'humidity_mean_pct',
                'ssta_elino', 'ssta_lanina', 'SOI_index', 'NAO_index'],
    'macro'  : ['DJIA_Index', 'WTI_Index', 'Broad_Dollar_index', 'Stock_Uncertainty'],
    'news'   : ['frbsf_sentiment', 'epu_index', 'Text_Climate_Anomaly'],
}

GROUP_COLORS = {
    'rv': '#e41a1c', 'climate': '#377eb8', 'macro': '#4daf4a', 'news': '#984ea3',
}

FEATURE_SETS = {
    'climate'              : ['climate'],
    'climate_rv'           : ['climate', 'rv'],
    'macro_news_climate_rv': ['macro', 'news', 'climate', 'rv'],
}

def build_cols(groups: list[str]) -> list[str]:
    cols: list[str] = []
    for g in groups:
        cols.extend(FEATURE_GROUPS[g])
    return cols

def group_of(col: str) -> str:
    for g, cols in FEATURE_GROUPS.items():
        if col in cols:
            return g
    return 'other'

for name, groups in FEATURE_SETS.items():
    print(f'{name:30s}: {len(build_cols(groups)):2d} vars')

climate                       : 14 vars
climate_rv                    : 15 vars
macro_news_climate_rv         : 22 vars


## 4. Preprocessing Pipeline

**Identical to `eda_preprocessing.ipynb`** — same 4-step pipeline used for ParCorr and GPDC.

| Step | Why it also matters for CMI |
|------|----------------------------|
| Impute (interpolate + ffill) | NaNs break k-NN distance computation |
| Gaussian detrend (15-yr) | Secular drift in CO₂/DJIA inflates CMI via spurious joint-density concentration |
| Anomalize (phase-wise) | Annual seasonal cycles create periodic density ridges that bias the k-NN radius estimate |
| `pp.trafo2normal` | Rank-based normalization reduces k-NN boundary effects and estimation bias for heavy-tailed variables |

In [5]:
def load_and_preprocess(crop: str, frequency: str) -> tuple[np.ndarray, list[str], pd.Series]:
    """Load CSV, impute, detrend, anomalize, Gaussianise. Returns (arr_normal, col_names, dates)."""
    path = ROOT / 'data' / frequency / f'{crop}.csv'
    raw  = pd.read_csv(path, parse_dates=['date'])
    df   = raw.sort_values('date').reset_index(drop=True)

    # Impute
    for col in ['pdsi', 'Broad_Dollar_index']:
        if col in df.columns:
            df[col] = df[col].interpolate(method='linear', limit_direction='both')
    for col in ['DJIA_Index']:
        if col in df.columns:
            df[col] = df[col].ffill().bfill()
    df = df.dropna().reset_index(drop=True)

    col_names = [c for c in df.columns if c != 'date']
    arr = df[col_names].values.astype(float)

    # Detrend (15-year Gaussian smoother)
    cycle_len    = 52 if frequency == 'weekly' else 12
    smooth_width = 15 * cycle_len
    arr_detrended = pp.smooth(arr, smooth_width=smooth_width, kernel='gaussian', residuals=True)

    # Anomalize
    arr_anom = arr_detrended.copy().astype(float)
    for phase in range(cycle_len):
        idx = slice(phase, None, cycle_len)
        arr_anom[idx] -= arr_detrended[idx].mean(axis=0)
        std = arr_detrended[idx].std(axis=0)
        std[std == 0] = 1.0
        arr_anom[idx] /= std

    # Gaussianise (rank-based van der Waerden scores)
    arr_normal = pp.trafo2normal(arr_anom)

    return arr_normal, col_names, df['date']


print('Preprocessing function defined.')

Preprocessing function defined.


## 4b. Preprocessed Data Overview

All plots below use data after the full pipeline (impute → detrend → anomalize → Gaussianise).

- **Time series** — all 22 variables for corn (representative crop) after preprocessing
- **Distributions** — marginal histograms vs N(0,1) reference; after `trafo2normal` all columns should approximate the red curve
- **Cross-correlations** — CCF of each variable X at lags 0..`tau_max` against `weekly_rv`, shown for all three crops simultaneously

In [ ]:
from matplotlib.patches import Patch as _Patch
from matplotlib.lines import Line2D as _Line2D

_raw_df    = pd.read_csv(ROOT / 'data' / FREQUENCY / 'corn.csv', parse_dates=['date'])
_raw_cols  = [c for c in _raw_df.columns if c != 'date']
_raw_arr   = _raw_df[_raw_cols].values.astype(float)
_raw_dates = _raw_df['date']

_arr0, _cols0, _dates0 = load_and_preprocess('corn', FREQUENCY)
_N   = len(_cols0)
_ncg = 4
_nrg = (_N + _ncg - 1) // _ncg

fig, axes = plt.subplots(_nrg, _ncg, figsize=(_ncg * 4, _nrg * 2.4), sharex=True)
axes = axes.flatten()
for i, col in enumerate(_cols0):
    ax = axes[i]
    color = GROUP_COLORS.get(group_of(col), '#888')
    if col in _raw_cols:
        _r     = _raw_arr[:, _raw_cols.index(col)].astype(float)
        _r_std = _r.std()
        _r_z   = (_r - _r.mean()) / (_r_std if _r_std > 0 else 1.0)
        ax.plot(_raw_dates, _r_z, lw=0.7, color='#bbbbbb', alpha=0.9, zorder=1)
    ax.plot(_dates0, _arr0[:, i], lw=0.9, color=color, alpha=0.85, zorder=2)
    ax.axhline(0, color='k', lw=0.3, ls='--', zorder=0)
    ax.set_title(col, fontsize=8, pad=2)
    ax.tick_params(labelsize=6)
    ax.set_ylabel('σ', fontsize=6)
    for sp in ['top', 'right']:
        ax.spines[sp].set_visible(False)
for ax in axes[_N:]:
    ax.set_visible(False)
_leg = [_Line2D([0],[0], color='#bbbbbb', lw=1.2, label='raw (z-scored for display)'),
        _Line2D([0],[0], color='steelblue', lw=1.2, label='preprocessed')]
_leg += [_Patch(facecolor=c, label=g) for g, c in GROUP_COLORS.items()]
fig.legend(handles=_leg, loc='lower center', ncol=3, fontsize=8, frameon=False)
fig.suptitle(
    f'Raw vs Preprocessed — Corn {FREQUENCY.capitalize()}\n'
    '(grey = raw z-scored for display  |  colour = after detrend + anomalize + Gaussianise)',
    fontsize=11, fontweight='bold',
)
plt.tight_layout()
plt.show()

In [ ]:
_preproc = {}
for _crop in CROPS:
    _a, _c, _d = load_and_preprocess(_crop, FREQUENCY)
    _preproc[_crop] = (_a, _c, _d)

_LAGS  = [0, 1, 2, 3]
_other = [c for c in _cols0 if c != 'weekly_rv']
_n_oth = len(_other)
_ci    = 1.96 / np.sqrt(_N)

fig, axes = plt.subplots(1, 3, figsize=(18, max(7, _n_oth * 0.38 + 1.5)), sharey=True)

for ax, crop in zip(axes, CROPS):
    arr, cols, _ = _preproc[crop]
    if 'weekly_rv' not in cols:
        continue
    rv = arr[:, cols.index('weekly_rv')]

    corr_mat = np.full((_n_oth, len(_LAGS)), np.nan)
    for li, lag in enumerate(_LAGS):
        for fi, feat in enumerate(_other):
            if feat not in cols:
                continue
            x = arr[:, cols.index(feat)]
            corr_mat[fi, li] = np.corrcoef(x[: len(x) - lag] if lag else x,
                                            rv[lag:] if lag else rv)[0, 1]

    im = ax.imshow(corr_mat, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
    ax.set_xticks(range(len(_LAGS)))
    ax.set_xticklabels([f'lag {l}' for l in _LAGS], fontsize=9)
    ax.set_yticks(range(_n_oth))
    ax.set_yticklabels(_other, fontsize=8)

    for fi in range(_n_oth):
        for li in range(len(_LAGS)):
            val = corr_mat[fi, li]
            if not np.isnan(val):
                ax.text(li, fi, f'{val:.2f}', ha='center', va='center',
                        fontsize=7, color='white' if abs(val) > 0.5 else 'black')

    ax.set_title(crop.capitalize(), fontsize=11, fontweight='bold')

plt.colorbar(im, ax=axes[-1], shrink=0.6, label='Pearson r')
fig.suptitle(
    f'Correlation: features → weekly_rv at lags 0–3 (preprocessed) | {FREQUENCY.capitalize()}\n'
    f'(corr(X(t−k), rv(t))  |  95% CI ± {_ci:.2f})',
    fontsize=11, fontweight='bold',
)
plt.tight_layout()
plt.show()

## 5. PCMCI + CMI Helper Functions

In [6]:
def run_cmi(
    feature_set_name: str,
    col_names: list[str],
    arr_preprocessed: np.ndarray,
    all_cols: list[str],
    tau_max: int,
    k: int = 5,
    sig_samples: int = 200,
    pc_alpha: float = 0.05,
    alpha_level: float = 0.05,
    fdr_method: str = 'fdr_bh',
    max_combinations: int = 1,
    max_conds_dim: int | None = None,
    verbosity: int = 0,
) -> dict:
    """
    Run PCMCI + FAISSCMI on a named feature subset.

    Re-instantiates FAISSCMI each call to reset GPU resources and internal state.
    max_combinations=1 tests only the single best conditioning set per MCI link —
    the primary runtime control when significance requires shuffle testing.
    """
    indices     = [all_cols.index(c) for c in col_names]
    data_subset = arr_preprocessed[:, indices]

    dataframe = pp.DataFrame(
        data_subset,
        var_names=col_names,
        datatime={0: np.arange(len(data_subset))},
    )
    cmi = FAISSCMI(
        k=k,
        use_gpu=FAISS_USE_GPU,
        gpu_device=0,
        standardize=True,
        jitter=1e-6,
        significance='shuffle_test',
        sig_samples=sig_samples,
        seed=RANDOM_SEED,
    )
    pcmci = PCMCI(dataframe=dataframe, cond_ind_test=cmi, verbosity=verbosity)

    print(f'\n  ── {feature_set_name} ({len(col_names)} vars, tau_max={tau_max}) ──')

    results = pcmci.run_pcmci(
        tau_min=1,
        tau_max=tau_max,
        pc_alpha=pc_alpha,
        alpha_level=alpha_level,
        fdr_method=fdr_method,
        max_combinations=max_combinations,
        max_conds_dim=max_conds_dim,
    )
    q_matrix = pcmci.get_corrected_pvalues(
        p_matrix=results['p_matrix'], tau_max=tau_max, fdr_method=fdr_method,
    )
    graph = pcmci.get_graph_from_pmatrix(
        p_matrix=q_matrix, alpha_level=alpha_level, tau_min=1, tau_max=tau_max,
    )
    results.update({
        'graph': graph, 'q_matrix': q_matrix,
        'pcmci': pcmci, 'col_names': col_names,
    })
    return results


def plot_results(results: dict, title: str, tau_max: int) -> None:
    col_names = results['col_names']
    n = len(col_names)

    tp.plot_graph(
        val_matrix=results['val_matrix'],
        graph=results['graph'],
        var_names=col_names,
        link_colorbar_label='cross-MCI (CMI)',
        node_colorbar_label='auto-MCI (CMI)',
        show_autodependency_lags=True,
        figsize=(max(8, n * 0.6), max(6, n * 0.5)),
    )
    plt.suptitle(f'{title}\ntau_max={tau_max} | FDR-BH alpha=0.05',
                 fontsize=11, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

    tp.plot_time_series_graph(
        val_matrix=results['val_matrix'],
        graph=results['graph'],
        var_names=col_names,
        link_colorbar_label='MCI (CMI)',
        figsize=(max(12, n * 0.8), 6),
    )
    plt.suptitle(f'{title} — time-series graph',
                 fontsize=11, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()


def extract_rv_links(results: dict, alpha: float = 0.05) -> pd.DataFrame:
    col_names = results['col_names']
    if 'weekly_rv' not in col_names:
        return pd.DataFrame()
    rv_idx  = col_names.index('weekly_rv')
    q_mat   = results['q_matrix']
    val_mat = results['val_matrix']
    rows = [
        {'cause': col_names[j], 'lag': tau, 'val': val_mat[j, rv_idx, tau],
         'q_val': q_mat[j, rv_idx, tau]}
        for j in range(len(col_names))
        for tau in range(1, q_mat.shape[2])
        if q_mat[j, rv_idx, tau] < alpha
    ]
    return pd.DataFrame(rows)


print('PCMCI + CMI helpers defined.')

PCMCI + CMI helpers defined.


## 6. Parameter Summary

| Parameter | Value | Why for FAISSCMI |
|-----------|-------|-----------------|
| `tau_max` | **3** | Each MCI test runs sig_samples shuffles; tau_max=3 caps link candidates and conditioning-set dimensions |
| `pc_alpha` | **0.05** | Tighter than ParCorr (0.2) — prunes PC skeleton more aggressively; fewer links enter the expensive shuffle-test MCI step |
| `k_neighbors` | **5** | Kraskov recommend k ∈ [3, 10]; k=5 balances bias and variance for T ≈ 192 |
| `sig_samples` | **200** | Shuffle iterations per test; 200 gives ≈ ±3 pp error on p-values near 0.05 |
| `max_combinations` | **1** | Single best conditioning set per MCI link — primary runtime control for shuffle-test methods |
| `max_conds_dim` | **None** | PC step's own pruning limits parent-set size; capping separately drops true parents |
| `significance` | **`shuffle_test`** | Only valid option — CMI has no analytic null |
| `alpha_level` | **0.05** | Final FDR-BH significance threshold |
| `jitter` | **1e-6** | Prevents k-NN ties that bias CMI upward |
| `standardize` | **True** | Uniform k-NN radius across variables with different scales |

## 7. Run All Crops

For each crop: load → preprocess → run 3 feature sets → print significant links → plot.

> **Runtime warning:** On CPU with sig_samples=200 at tau_max=3:
> - `climate` (10 vars): ~2–5 min per crop
> - `climate_rv` (11 vars): ~3–7 min per crop
> - `macro_news_climate_rv` (18 vars): ~10–30 min per crop
>
> Total for 3 crops × 3 feature sets: **1–2 hours on CPU**.
> For a quick exploratory run, set `SIG_SAMPLES = 50` and `TAU_MAX = 2` in cell 0.

In [ ]:
import time

all_results: dict[str, dict] = {}

for crop in CROPS:
    sep = '#' * 70
    print(f'\n{sep}')
    print(f'  CROP: {crop.upper()}  |  FREQUENCY: {FREQUENCY.upper()}')
    print(sep)

    arr_normal, all_cols, dates = load_and_preprocess(crop, FREQUENCY)
    print(f'  Preprocessed: {arr_normal.shape}  '
          f'({dates.min().date()} -> {dates.max().date()})')

    all_results[crop] = {}

    for fs_name, groups in FEATURE_SETS.items():
        col_names = [c for c in build_cols(groups) if c in all_cols]

        t0  = time.time()
        res = run_cmi(
            feature_set_name=fs_name,
            col_names=col_names,
            arr_preprocessed=arr_normal,
            all_cols=all_cols,
            tau_max=TAU_MAX,
            k=K_NEIGHBORS,
            sig_samples=SIG_SAMPLES,
            pc_alpha=PC_ALPHA,
            alpha_level=ALPHA_LEVEL,
            fdr_method=FDR_METHOD,
            max_combinations=MAX_COMBINATIONS,
            max_conds_dim=None,
            verbosity=0,
        )
        elapsed = time.time() - t0
        all_results[crop][fs_name] = res

        print(f'  Elapsed: {elapsed:.1f} s')
        res['pcmci'].print_significant_links(
            p_matrix=res['q_matrix'],
            val_matrix=res['val_matrix'],
            alpha_level=ALPHA_LEVEL,
        )
        plot_results(
            res,
            title=f'{crop.capitalize()} {FREQUENCY} — {fs_name} — PCMCI + FAISS CMI',
            tau_max=TAU_MAX,
        )

print('\nAll crops done.')


######################################################################
  CROP: CORN  |  FREQUENCY: MONTHLY
######################################################################
  Preprocessed: (192, 22)  (2009-04-01 -> 2025-03-01)

  ── climate (14 vars, tau_max=3) ──


## 8. Cross-Crop Comparison: Significant Links into `weekly_rv`

Which variables Granger-cause realized volatility for each crop, from both RV feature sets.

In [ ]:
for fs_name in ['climate_rv', 'macro_news_climate_rv']:
    sep = '=' * 65
    print(f'\n{sep}')
    print(f'  Feature set: {fs_name}')
    print(sep)
    for crop in CROPS:
        links = extract_rv_links(all_results[crop][fs_name])
        print(f'\n  {crop.upper()}:')
        if links.empty:
            print('    No significant links into weekly_rv.')
        else:
            print(links.sort_values('q_val').to_string(
                index=False, float_format='{:.3f}'.format, col_space=18))

In [ ]:
for fs_name in ['climate_rv', 'macro_news_climate_rv']:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)

    for ax, crop in zip(axes, CROPS):
        links = extract_rv_links(all_results[crop][fs_name])
        if links.empty:
            ax.set_title(f'{crop.capitalize()}\n(no significant links)', fontsize=10)
            continue
        labels = [f"{r['cause']} (tau={r['lag']})" for _, r in links.iterrows()]
        vals   = links['val'].values
        colors = [GROUP_COLORS.get(group_of(r['cause']), '#888') for _, r in links.iterrows()]
        ax.barh(labels, vals, color=colors, alpha=0.85)
        ax.axvline(0, color='k', lw=0.8)
        ax.set_xlabel('MCI (CMI)')
        ax.set_title(f'{crop.capitalize()}', fontsize=11, fontweight='bold')
        for spine in ['top', 'right']:
            ax.spines[spine].set_visible(False)

    handles = [Patch(facecolor=c, label=g) for g, c in GROUP_COLORS.items()]
    fig.legend(handles=handles, loc='lower center', ncol=4, fontsize=9)
    fig.suptitle(
        f'Significant causal links into weekly_rv — {fs_name}\n'
        f'{FREQUENCY.capitalize()} | PCMCI + FAISS CMI | tau_max={TAU_MAX} | FDR-BH alpha=0.05',
        fontsize=11, fontweight='bold',
    )
    plt.tight_layout()
    plt.show()

## 9. CMI vs ParCorr: Unique Links into `weekly_rv`

Links that FAISSCMI finds but ParCorr misses indicate **nonlinear or non-Gaussian** dependencies.
Links ParCorr finds that CMI misses indicate insufficient CMI power at k=5, T≈192 for that linear link.

In [ ]:
print('Granger-causes of weekly_rv — FAISSCMI (climate_rv, tau <= 3):')
print(f'{"Crop":<12}  {"Cause":<35}  {"Lag":<5}  {"MCI (CMI)":<12}  {"q-val":<8}')
print('-' * 80)

for crop in CROPS:
    links = extract_rv_links(all_results[crop]['climate_rv'])
    if links.empty:
        print(f'{crop:<12}  (none)')
    else:
        for _, row in links.sort_values('q_val').iterrows():
            print(f"{crop:<12}  {row['cause']:<35}  "
                  f"tau={row['lag']:1d}  {row['val']:+.4f}        {row['q_val']:.4f}")

## 10. Summary

**Preprocessing (identical to `eda_preprocessing.ipynb`):**
- Interpolate `pdsi`, `Broad_Dollar_index` | ffill `DJIA_Index`
- `pp.smooth(smooth_width=180, kernel='gaussian', residuals=True)` — 15-yr Gaussian detrend
- Phase-wise anomalization (`cycle_length=12`)
- `pp.trafo2normal` — rank-based Gaussianisation

**PCMCI + FAISSCMI config:**

| Parameter | Value | Reason |
|-----------|-------|--------|
| `tau_max` | 3 | Shuffle tests scale linearly with link count; tau=3 balances detection range and runtime |
| `pc_alpha` | 0.05 | Tighter skeleton — reduces shuffle-test overhead in MCI step |
| `k_neighbors` | 5 | Robust Kraskov choice for T ≈ 192 |
| `sig_samples` | 200 | Standard permutation precision; reduce to 50 for exploratory runs |
| `max_combinations` | 1 | Single conditioning set per MCI link — essential runtime control |
| `significance` | `shuffle_test` | Only valid option for CMI |
| `alpha_level` | 0.05 | Final FDR-BH threshold |

**Interpreting CMI values:**
- CMI ≥ 0 theoretically; small negative `val_matrix` entries are k-NN estimation noise for null links
- CMI finds, ParCorr misses → nonlinear / non-Gaussian dependency
- CMI finds, GPDC misses → dependency violates GPDC's additive-noise assumption (e.g., multiplicative)
- ParCorr finds, CMI misses → CMI had insufficient power at k=5, T=192 for that linear link

**Next steps:** Overlay CMI, ParCorr, and GPDC graphs to classify each link as linear-only, nonlinear-additive, or fully general.